# Phase 2 – Model: dengue or not dengue based on symptoms?

This notebook **explains and visualizes** the training. The logic lives in `src/train.py` and
`src/preprocessing.py`; the same functions are imported here, so the notebook and the script always give the same results.
The official model (`models/model.pkl`) is generated with:

```bash
python -m src.train
```

### Summary of decisions (approved in Phase 1)
| Decision | Choice | Why |
|---|---|---|
| Target variable | 1 = confirmed dengue, 0 = discarded (`ajuste_ = 6`); "probable" cases are excluded | Only cases with a reliable label are used |
| Input variables | age group, sex and 10 reportable symptoms | What a patient can fill in on a form |
| Excluded | fever (constant), laboratory/exam (platelets, hematocrit, enlarged liver…), leakage (shock, organ damage, final classification, hospitalization…) | No information, not reportable, or data leakage |

In [ ]:
import sys
sys.path.insert(0, "..")  # project root, to import the src package

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, precision_score, recall_score

from src.preprocessing import FEATURES, prepare_data
from src.train import METRICS, run_training

pd.set_option("display.precision", 3)

## 1. Data and cleaning
- Symptoms: SIVIGILA uses **1 = Yes, 2 = No**. They are normalized to **1/0** with `to_binary()`. Any unrecognized
  value becomes null.
- Nulls: imputed with the **most frequent value** *inside* the pipeline. This way the imputation is learned only from the
  training data (no data leakage) and the web app applies it the same way to an incomplete form.
- Age and sex: **one-hot** encoding. A linear model must not treat the age group "60+" as if it were "6 times 0-5".

In [ ]:
X, y, dates = prepare_data()
print(f"Cases: {len(X):,}  |  dengue: {y.mean():.1%}  |  features: {FEATURES}")
print("\nNulls per feature:\n", X.isna().sum().to_string())
X.head()

## 2. Training and model comparison
- **Split:** 80 % training / 20 % test, **stratified** (same dengue share in both). The test set
  is not used until the end.
- **5-fold stratified cross-validation** on the 80 %. Each model is trained 5 times on
  different portions and the metrics are averaged. It is more reliable than a single split.
- **Imbalance (68/32):** `class_weight="balanced"` gives more weight to errors in the minority class. It does not create
  synthetic data (SMOTE) nor remove data (undersampling).
- **Models:**
  1. *Logistic Regression*: linear, interpretable, serves as a baseline.
  2. *Random Forest*: many trees voting; captures interactions between symptoms.
  3. *Gradient Boosting* (`HistGradientBoostingClassifier`): sequential trees that correct the errors of the
     previous one. It is scikit-learn's implementation, equivalent to XGBoost, and adds no extra dependency to the deployment.

In [ ]:
artifact, ctx = run_training(save=False)

In [ ]:
cv = pd.DataFrame(artifact["cv_metrics"]).set_index("model")
cv[METRICS]

In [ ]:
ax = cv[METRICS].T.plot.bar(figsize=(9, 4), rot=0)
ax.set_ylim(0, 1); ax.set_title("Cross-validation (5 folds, threshold 0.5)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

**Reading:**
- The three models are **very close** (ROC-AUC between 0.63 and 0.64). When models this different give the same result,
  the limit is **the information in the data** and not the algorithm.
- **Selection criterion:** the highest mean ROC-AUC. It is independent of the threshold and measures how well the model *ranks*
  patients by risk. If the difference were smaller than 0.005, the simplest model would be chosen.

In [ ]:
print("Selected model:", artifact["model_name"])
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, model in ctx["models"].items():
    model.fit(ctx["X_train"], ctx["y_train"])
    RocCurveDisplay.from_estimator(model, ctx["X_test"], ctx["y_test"], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Chance (AUC 0.5)"); ax.legend(loc="lower right")
ax.set_title("ROC curves on the test set"); plt.tight_layout(); plt.show()

## 3. Decision threshold: prioritize dengue recall
By default a classifier says "dengue" if the probability is ≥ 0.5. With that threshold the recall is only ~55 %:
**almost half of the dengue cases would slip through**. In public health a false negative (sending someone with dengue
home) is worse than a false positive (ordering one extra test).

**Rule used:** the **highest** threshold that achieves **recall ≥ 80 %**, computed with *out-of-fold* predictions on the
training set (never with the test set).

Why not maximize F2 or lower the threshold to 0? With 68 % positives, a "model" that says *dengue* to everyone
has 100 % recall and F2 ≈ 0.91, but it is useless. The minimum-recall rule avoids that trap.

In [ ]:
model = ctx["models"][artifact["model_name"]]
proba = model.predict_proba(ctx["X_test"])[:, 1]
thresholds = np.arange(0.2, 0.8, 0.01)
curves = pd.DataFrame({
    "recall (dengue)": [recall_score(ctx["y_test"], proba >= u) for u in thresholds],
    "precision": [precision_score(ctx["y_test"], proba >= u, zero_division=0) for u in thresholds],
    "specificity": [recall_score(1 - ctx["y_test"], proba < u) for u in thresholds],
}, index=thresholds)
ax = curves.plot(figsize=(8, 4))
ax.axvline(artifact["threshold"], color="k", ls="--", label=f"selected threshold = {artifact['threshold']}")
ax.axvline(0.5, color="gray", ls=":", label="default threshold = 0.5")
ax.set_xlabel("Threshold"); ax.set_title("Recall / specificity trade-off (test)"); ax.legend()
plt.tight_layout(); plt.show()

## 4. Final results on the test set (20 % never seen)

In [ ]:
rows = ["accuracy", "precision", "recall", "specificity", "f1", "roc_auc"]
pd.DataFrame({
    f"selected threshold ({artifact['threshold']})": {k: artifact["test_metrics"][k] for k in rows},
    "threshold 0.5": {k: artifact["test_metrics_threshold_05"][k] for k in rows},
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (title, key) in zip(axes, [(f"Threshold {artifact['threshold']}", "test_metrics"), ("Threshold 0.5", "test_metrics_threshold_05")]):
    ConfusionMatrixDisplay(np.array(artifact[key]["confusion_matrix"]),
                           display_labels=["Not dengue", "Dengue"]).plot(ax=ax, cmap="Reds", colorbar=False)
    ax.set_title(title)
plt.tight_layout(); plt.show()

**Reading the confusion matrix (selected threshold):**
- ~**81 % of real dengue cases** are detected (true positives), versus ~55 % with the 0.5 threshold.
- The cost: many false positives. Only ~1 in 3 patients without dengue is correctly classified (specificity ≈ 0.34).
- **Correct interpretation:** it is a **screening** tool ("it is worth running the dengue test"),
  **not a diagnostic** one. This matches the mandatory notice in the app.

## 5. Feature importance
**Permutation importance:** one variable is randomly shuffled in the test set and the drop in ROC-AUC is measured.
If it drops a lot, the model depends on that variable. It works the same for any model and operates on the
original variables (age as a whole, not each one-hot column separately).

In [ ]:
imp = pd.DataFrame(artifact["importance"]).sort_values("importance")
ax = imp.plot.barh(x="label", y="importance", xerr="std", legend=False, figsize=(7, 5), color="#c0392b")
ax.set_xlabel("ROC-AUC drop when permuted"); ax.set_ylabel("")
ax.set_title(f"Feature importance ({artifact['model_name']})")
plt.tight_layout(); plt.show()

In [ ]:
# Interpretable complement: Logistic Regression coefficients (sign = direction of the effect)
lr = ctx["models"]["Logistic Regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr.named_steps["clf"].coef_[0], index=[n.split("__")[1] for n in names])
coef.sort_values().plot.barh(figsize=(7, 6), color=np.where(coef.sort_values() > 0, "#c0392b", "#7f8c8d"))
plt.axvline(0, color="k", lw=0.8); plt.title("Logistic Regression: coefficients (positive = more likely dengue)")
plt.tight_layout(); plt.show()

**Reading:** the variables that contribute the most are the **age group**, **skin rash** and **abdominal pain**,
followed by headache and retro-orbital pain. This is consistent with Phase 1 and with the clinical description of dengue.
Muscle pain, joint pain and drowsiness contribute almost nothing, because they also appear in other viral infections.

## 6. Temporal validation (robustness test)
A random split mixes cases from all years. Here we test whether the model works in a **period it has not seen**.

In [ ]:
pd.DataFrame(artifact["temporal_validation"])

**Reading (important limitation):**
- Trained on 2024 and evaluated on 2023, the ROC-AUC holds (~0.63). The signal is real and stable across those years.
- In **2025–2026** the ROC-AUC drops to ~0.50 (chance). In that post-epidemic period there are very few confirmed dengue cases
  (~20 %), and the relationship between symptoms and diagnosis disappears or reverses. Probable causes are other
  viral infections circulating with similar symptoms and cases not yet updated in SIVIGILA.
- **Conclusion:** the model reflects the 2023–2024 pattern and **should be retrained periodically** with new data.

## 7. Saved model
`python -m src.train` saves a dictionary to `models/model.pkl` with:
- `model`: the full pipeline (imputation + one-hot + classifier), **retrained with all the data**.
  The reported metrics are those of the test set, which the model had not seen.
- `features`, `threshold`, `age_groups`, `symptom_labels`: what the web app needs to build the form.
- Metrics, importance and scikit-learn version, for the "About the model" page.

It also saves `models/metrics.json`, a readable version without the model.

## 8. Conclusions for the presentation
1. With reportable symptoms only, a **ROC-AUC ≈ 0.63–0.64** is achieved: better than chance, but modest. This is expected,
   because **dengue is confirmed with laboratory tests precisely because symptoms are not enough**.
2. The three algorithms perform the same, so the limit is in the data, not the model.
3. The threshold was tuned to detect **~80 % of dengue cases**, at the cost of false alarms. This is suitable for **screening**.
4. Limitations: age only in ranges, data from a single municipality and temporal drift in 2025–2026.